# 🧪 PT-W1-D1 概念实验：用 dataclass 模拟 Bounded Context 划分

> 配套阅读：同名 .md
> 实验目标：用代码模拟 MI 的 17 个 Bounded Context，验证 Context 间概念歧义

## 第 1 格：17 个 Bounded Context 的优先级分布

In [ ]:
from dataclasses import dataclass, field
from typing import List

@dataclass
class BoundedContext:
    name: str
    priority: str
    core_objects: List[str]
    ddd_score: int

contexts = [
    BoundedContext("Property Management", "P0", ["Project","Building","Floor","ResourceUnit"], 4),
    BoundedContext("Space Management", "P0", ["ResourceUnit","ResourceType","PhysicalAttributes"], 4),
    BoundedContext("Lease Management", "P0", ["Occupancy","OccupancyPeriod","VacancyRate"], 5),
    BoundedContext("Contract Management", "P0", ["Contract","ContractClause","RentMethod"], 4),
    BoundedContext("Billing Management", "P0", ["Bill","ChargeDefinition","BillingRule"], 4),
    BoundedContext("Payment Management", "P0", ["Payment","PaymentRecord"], 3),
    BoundedContext("Finance Settlement", "P0", ["Settlement","SettlementConfig"], 3),
    BoundedContext("Tenant Management", "P0", ["Tenant","LegalIdentity"], 3),
    BoundedContext("Customer Management", "P0", ["Customer","Lead"], 3),
    BoundedContext("Sales/Mall Operations", "P1", ["SalesRecord","MallEvent"], 2),
    BoundedContext("Marketing/Promotion", "P1", ["Campaign","PromotionRule"], 2),
    BoundedContext("Property Service", "P1", ["WorkOrder","Inspection"], 2),
    BoundedContext("Asset Evaluation", "P1", ["Assessment","Depreciation"], 2),
    BoundedContext("Energy Management", "P1", ["Meter","EnergyRecord"], 2),
    BoundedContext("Park Operations", "P1", ["ParkRule","ParkingRecord"], 2),
    BoundedContext("Report/Analytics", "P1", ["Dashboard","Report"], 1),
    BoundedContext("System/Admin", "P1", ["User","Permission","AuditLog"], 1),
]

p0 = [c for c in contexts if c.priority == "P0"]
p1 = [c for c in contexts if c.priority == "P1"]
print(f"Core Domain (P0): {len(p0)} 个 Context")
print(f"Supporting Domain (P1): {len(p1)} 个 Context")
for c in p0:
    print(f"  {c.name:<24} objects={c.core_objects}")

## 第 2 格：同一概念在不同 Context 中的语义歧义

DDD 核心检验：一个概念在不同 Context 里可以有不同的含义。

In [ ]:
concept_space = {
    "Space Management": "铺位的物理属性：面积、楼层、工程条件、资源类型",
    "Lease Management": "可租赁单元：租金、租期、是否可出租",
    "Property Service": "维护对象：设备清单、工单关联、巡检目标",
    "Asset Evaluation": "评估标的：资产价值、折旧、重估",
}

for ctx, meaning in concept_space.items():
    print(f"\n[{ctx}]")
    print(f"  Space = {meaning}")

print("\n⚠️ 如果四个 Context 共用一个 Space 模型，说明 Context 边界没划好。")
print("DDD 要求：每个 Context 内保持一致的语义模型，跨 Context 通过翻译层交互。")

## 第 3 格：DDD 对照 — Domain Model 完成度

In [ ]:
ddd_checklist = [
    ("Domain", "商业地产运营管理", "✅ 完整"),
    ("Bounded Context", "17 个 Context", "✅ 边界已划分"),
    ("Core Domain", "P0 的 9 个 Full Model", "✅ 已区分"),
    ("Supporting Domain", "P1 的 8 个", "✅"),
    ("Context Map", "跨 Context 集成契约", "⚠️ 不够显式"),
    ("Ubiquitous Language", "域知识 + 术语表", "⚠️ 部分有"),
    ("Ontology（语义层）", "——", "❌ 缺失"),
]

print(f"{'DDD 概念':<20} {'你的对应物':<28} {'符合度':<10}")
print("-" * 60)
for concept, mapping, status in ddd_checklist:
    print(f"{concept:<20} {mapping:<28} {status:<10}")

score = sum(1 for _,_,s in ddd_checklist if "✅" in s)
total = len(ddd_checklist)
print(f"\nDDD 战略设计完成度：{score}/{total} ≈ {score*100//total}%")
print("\n核心结论：Domain Model ≠ Ontology")
print("  Domain Model = 建筑施工图（给工程师看）")
print("  Ontology    = 建筑说明书（给 AI 看）")

## 第 4 格：可视化 — Bounded Context 优先级分布

In [ ]:
from matplotlib import font_manager, pyplot as plt
import numpy as np

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False
print("字体:", font_name)
import matplotlib.patches as mpatches

labels = [c.name for c in contexts]
priorities = [0 if c.priority == "P0" else 1 for c in contexts]
scores = [c.ddd_score for c in contexts]

fig, ax = plt.subplots(figsize=(10, 6))
colors = ["#e74c3c" if p == "P0" else "#3498db" for p in priorities]
ax.barh(range(len(labels)), scores, color=colors, alpha=0.8)
ax.set_yticks(range(len(labels)))
ax.set_yticklabels(labels, fontsize=10)
ax.set_xlabel("DDD 完成度评分", fontsize=12)
ax.set_title("MI 17 个 Bounded Context 优先级与 DDD 成熟度", fontsize=14)
ax.invert_yaxis()
ax.legend(handles=[mpatches.Patch(facecolor="#e74c3c",label="P0 Core"),
                    mpatches.Patch(facecolor="#3498db",label="P1 Supporting")])
plt.tight_layout()
plt.savefig("/tmp/w1d1_contexts.png", dpi=120)
plt.show()
print("结论：你已经做了 DDD，Domain Model 是 Ontology 的基础但不是替代品")